In this notebook I will generate the conformers that will be analyzed by the next 2

In [1]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from rdkit import Chem
from scripts.cheminformatics_utils import generate_conformers, write_xyz, xyz_to_sdf

OUT_DIR = "../data/conformers_amines"
OPENBABEL_PATH = (
    "/opt/homebrew/bin/obabel"  # replace this path with your Open Babel installation
)

# Retrieve amines

In [2]:
df_amines = pd.read_csv(
    "/Users/aprunotto/Documents/alessio-code/sterics-github/sterics-descriptors/data/selected_amines.csv"
)
df_amines

,smiles,chembl_id,has_single_amine,amine_type,logP,MW,HDonors,HAcceptors
0,C[C@@H](N)Cc1c2ccoc2c(Br)c2ccoc12.Cl,CHEMBL6706,True,primary,4.2530,330.609,1,3
1,C[C@@H]1O[C@H](n2cnc(C(N)=O)n2)[C@@H](O)[C@H]1O,CHEMBL274535,True,primary,-1.9839,228.208,3,6
2,CC[C@H]1Cc2cc3c(C(F)(F)F)cc(O)nc3cc2N[C@@H]1CC,CHEMBL6959,True,secondary,4.7320,324.346,2,3
3,CC(C)Nc1nc2ccccc2n2cnnc12,CHEMBL269318,True,secondary,2.0978,227.271,1,4
4,CCOC(=O)C1=C(O)C(=O)N(c2cccc3ccccc23)C1,CHEMBL7013,True,tertiary,2.5616,297.310,1,4
...,...,...,...,...,...,...,...,...
295,CN(N=O)c1cc(O)ccc1O,CHEMBL269733,True,tertiary,1.2154,168.152,2,4
296,O=C(N[C@@H](Cc1cccs1)C(=O)O)c1ccccc1Br,CHEMBL7742,True,secondary,2.9363,354.225,2,3
297,Nc1ncnc2c1ncn2Cc1ccccc1,CHEMBL266094,True,primary,1.4568,225.255,1,4
298,CN1CCC[C@H]1c1cccnc1,CHEMBL3,True,tertiary,1.8483,162.236,0,2


# Generate conformers

### Generate conformers with rdkit

 - ~2 mins for 300 molecules

In [3]:
mols_with_confs = []

for smiles in tqdm(df_amines.smiles, desc="Generating conformers for amines"):
    mol_with_confs = generate_conformers(Chem.MolFromSmiles(smiles))
    mols_with_confs.append(mol_with_confs)

Generating conformers for amines: 100%|██████████| 300/300 [02:03<00:00,  2.43it/s]


### Save conformers to `.xyz` files

In [4]:
os.makedirs(OUT_DIR, exist_ok=True)

for idx, mol in enumerate(mols_with_confs):
    if mol:
        for conf_id in range(mol.GetNumConformers()):
            file_path = os.path.join(OUT_DIR, f"mol_{idx:03d}_conf_{conf_id+1}.xyz")
            write_xyz(mol, conf_id, file_path)

 - for each molecule, convert 1 xyz to sdf (will be needed to identify the nitrogen of the amine)

In [5]:
for xyz_file in tqdm(
    [
        os.path.join(OUT_DIR, file_name)
        for file_name in os.listdir(OUT_DIR)
        if file_name.endswith("_conf_1.xyz")
    ],
    desc="Converting XYZ to SDF",
):
    xyz_file_path = Path(xyz_file)
    sdf_file_path = xyz_file_path.with_suffix(".sdf")
    xyz_to_sdf(xyz_file_path, sdf_file_path, OPENBABEL_PATH)

Converting XYZ to SDF:   0%|          | 0/300 [00:00<?, ?it/s]

1 molecule converted
Converting XYZ to SDF:  40%|████      | 121/300 [00:18<00:26,  6.88it/s]==============================
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ../data/conformers_amines/mol_020_conf_1.xyz)

1 molecule converted
Converting XYZ to SDF:  47%|████▋     | 142/300 [00:21<00:22,  6.92it/s]==============================
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ../data/conformers_amines/mol_193_conf_1.xyz)

1 molecule converted
Converting XYZ to SDF:  48%|████▊     | 144/300 [00:21<00:22,  6.93it/s]==============================
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ../data/conformers_amines/mol_232_conf_1.xyz)

1 molecule converted
Converting XYZ to SDF:  77%|███████▋  | 230/300 [00:34<00:10,  6.64it/s]==============================
